<a href="https://colab.research.google.com/github/ZorahSai/Projeto-WebAgriculture-/blob/main/Coleta%20de%20Dados/%20passo_a_passo_fluxo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Objetivo: Construir pares:


```
ORIGEM                               DESTINO
Probe/Anchor que mede   ───────►    Anchor escolhida
      prb_id                         IP/FQDN
```



## PASSO 1 - Entender e testar a raiz da API

* Vamos apenas testar a API para verificar se conseguimos acessa-la.
* Usamos a requição GET
* As consultas públicas não exigem chave

###Importar biblioteca

In [1]:
import requests

###A URL base da API

In [2]:
BASE_URL = "https://atlas.ripe.net/api/v2"

###Consultar se a API Root esta acessível

In [3]:
url = f"{BASE_URL}/root/"

response = requests.get(
    url,
    timeout=30
)

print("Status HTTP:", response.status_code)

Status HTTP: 200


Pipeline:
```
Notebook -> HTTP GET --> RIPE Atlas API -->  HTTP 200 --> API acessível
```



###Visualizando a resposta da API em JSON

In [4]:
dados = response.json()
dados

{'measurements': 'https://atlas.ripe.net/api/v2/measurements/',
 'probes': ['https://atlas.ripe.net/api/v2/probes/',
  {'tags': 'https://atlas.ripe.net/api/v2/probes/tags/'}],
 'anchors': 'https://atlas.ripe.net/api/v2/anchors/',
 'anchor-measurements': 'https://atlas.ripe.net/api/v2/anchor-measurements/'}

###Estes são os recusros necessários para o projeto

In [5]:
print("Measurements:")
print(dados["measurements"])

print("\nAnchors:")
print(dados["anchors"])

print("\nAnchor Probes:")
print(dados["probes"])

Measurements:
https://atlas.ripe.net/api/v2/measurements/

Anchors:
https://atlas.ripe.net/api/v2/anchors/

Anchor Probes:
['https://atlas.ripe.net/api/v2/probes/', {'tags': 'https://atlas.ripe.net/api/v2/probes/tags/'}]


### Informações rápidas:
* `/api/v2/measurements/`: lista as measurements publicas existentes. Mostra todas as medições publicas (origem-destino).
* `/api/v2/anchors/`: lista e detalhe das anchors.
* `/api/v2/probes`: lista e busca as probes

###**Conclusão do PASSO 1**

Os principais recursos identificados são:

- `/measurements/`: medições existentes;
- `/probes/`: sondas que executam medições;
- `/anchors/`: Anchors do RIPE Atlas;
- `/anchor-measurements/`: relações envolvendo medições de Anchors.

Neste projeto não serão criadas novas medições.
O objetivo é localizar medições públicas existentes e selecionar
pares origem-destino para coleta. Até aqui não escolhemos nenhum par na rede.


##PASSO 2: Descobrir e endenter as Anchors disponíveis

* Neste passo vamos bucas as Anchors públicas/ativas e quais podemos usar como destino.
* A ideia é buscar anchors mais estáveis.
* Vamos usar country como parâmetro de busca.
* Temos que excluir as anchors desativadas com `is_disabled=false`.

###Criar a URL de Anchors

In [6]:
url_anchors = f"{BASE_URL}/anchors/"
url_anchors

'https://atlas.ripe.net/api/v2/anchors/'

>`anchors`: são probes mais robustos que podem atuar como probes (origem) e probes(destino). Possuem um `probe_id` e os metadados como nome, cidade, país, se esta ativa.

###Definindo os filtros

Podemos começar buscando anchors no Brasil. BR segue o código ISO de país.

In [7]:
params = {
    "country": "BR",
    "is_disabled": "false"
}

###Fazendo a requisição

In [8]:
response = requests.get(
    url_anchors,
    params=params,
    timeout=30
)

print("Status HTTP: ", response.status_code)

Status HTTP:  200


###Confimando a URL enviada

* Observe que o request transformou os parâmetros em uma query string para consulta na api.

In [9]:
print(response.url)

https://atlas.ripe.net/api/v2/anchors/?country=BR&is_disabled=false


###Convertendo a resposta em JSON

In [10]:
dados_anchors = response.json()
dados_anchors

{'count': 15,
 'next': None,
 'previous': None,
 'results': [{'id': 2334,
   'type': 'Anchor',
   'hostname': 'br-bhz-as10417-client',
   'fqdn': 'br-bhz-as10417-client.anchors.atlas.ripe.net',
   'probe': 6891,
   'is_ipv4_only': False,
   'ip_v4': '150.164.1.222',
   'as_v4': 10417,
   'ip_v4_gateway': None,
   'ip_v4_netmask': None,
   'ip_v6': '2001:12f0:601:a906::222',
   'as_v6': 10417,
   'ip_v6_gateway': None,
   'ip_v6_prefix': None,
   'city': 'Belo Horizonte',
   'country': 'BR',
   'company': 'Universidade Federal de Minas Gerais',
   'nic_handle': '',
   'geometry': {'type': 'Point', 'coordinates': [-43.9677359, -19.870682]},
   'tlsa_record': '',
   'is_disabled': False,
   'is_replacement': True,
   'replaces_anchor': 6300,
   'date_live': '2020-11-09T18:20:00',
   'date_decommissioned': None,
   'hardware_version': 99},
  {'id': 4093,
   'type': 'Anchor',
   'hostname': 'br-cci-as61618',
   'fqdn': 'br-cci-as61618.anchors.atlas.ripe.net',
   'probe': 7457,
   'is_ipv4_o

###Analisando a resposta

* dados_anchors é um dicionário Json e as chaves são `count`, `next`, `previous`, `results`
* a chave mais importante para o projeto é `results`

In [11]:
dados_anchors.keys()

dict_keys(['count', 'next', 'previous', 'results'])

###Quantidade de Anchors encontradas

In [12]:
print("Quantidade total:", dados_anchors["count"])

Quantidade total: 15


###Qual é a primeira anchor

In [13]:
primeira_anchor = dados_anchors["results"][0]
primeira_anchor

{'id': 2334,
 'type': 'Anchor',
 'hostname': 'br-bhz-as10417-client',
 'fqdn': 'br-bhz-as10417-client.anchors.atlas.ripe.net',
 'probe': 6891,
 'is_ipv4_only': False,
 'ip_v4': '150.164.1.222',
 'as_v4': 10417,
 'ip_v4_gateway': None,
 'ip_v4_netmask': None,
 'ip_v6': '2001:12f0:601:a906::222',
 'as_v6': 10417,
 'ip_v6_gateway': None,
 'ip_v6_prefix': None,
 'city': 'Belo Horizonte',
 'country': 'BR',
 'company': 'Universidade Federal de Minas Gerais',
 'nic_handle': '',
 'geometry': {'type': 'Point', 'coordinates': [-43.9677359, -19.870682]},
 'tlsa_record': '',
 'is_disabled': False,
 'is_replacement': True,
 'replaces_anchor': 6300,
 'date_live': '2020-11-09T18:20:00',
 'date_decommissioned': None,
 'hardware_version': 99}

Podemos observar que results retorna muita informação. As principais para este projeto são: `id` identifica a Anchor, `fqdn` fornece seu nome estável, `ip_v4` fornece o endereço IPv4, `city e `country` localizam o destino e `probe` identifica a probe hospedada naquela Ancho

###Mostrando apenas o que nos interessa

In [14]:
print("Anchor ID:", primeira_anchor.get("id"))
print("FQDN:", primeira_anchor.get("fqdn"))
print("IPv4:", primeira_anchor.get("ip_v4"))
print("País:", primeira_anchor.get("country"))
print("Cidade:", primeira_anchor.get("city"))
print("Probe hospedada:", primeira_anchor.get("probe"))

Anchor ID: 2334
FQDN: br-bhz-as10417-client.anchors.atlas.ripe.net
IPv4: 150.164.1.222
País: BR
Cidade: Belo Horizonte
Probe hospedada: 6891


Se considerarmos este anchor como DESTINO então:
```
ANCHOR
│
├── id        = identificador da Anchor
├── fqdn      = nome do equipamento
├── ip_v4     = endereço IPv4
├── country   = BR
├── city      = cidade
└── probe     = probe que está dentro dessa Anchor
```

* Importante observar que `anchor_id` é diferentes de `probe_id`. Não interprete que são duas anchors. Apenas que:
```
Anchor ID (2334) -- hospeda --> Probe ID (6891)
```

* Isto ocorre porque uma Anchor tem duas características importantes. Ela pode ser:
```
ORIGEM
Anchor/Probe ───────────► outro destino
```

```
DESTINO
outra Probe ───────────► Anchor
```








###Listando 5 Anchors do Brasil

In [15]:
for anchor in dados_anchors["results"][:5]:
    print(
        anchor.get("id"),
        "|",
        anchor.get("country"),
        "|",
        anchor.get("city"),
        "|",
        anchor.get("fqdn"),
        "|",
        anchor.get("ip_v4"),
        "| probe:",
        anchor.get("probe")
    )

2334 | BR | Belo Horizonte | br-bhz-as10417-client.anchors.atlas.ripe.net | 150.164.1.222 | probe: 6891
4093 | BR | Camaçari | br-cci-as61618.anchors.atlas.ripe.net | 170.81.100.114 | probe: 7457
3679 | BR | Feira de Santana | br-fec-as264479.anchors.atlas.ripe.net | 138.117.32.11 | probe: 7307
3497 | BR | Fortaleza | br-for-as37468.anchors.atlas.ripe.net | 170.238.235.214 | probe: 7242
2849 | BR | Ponta Grossa | br-pgz-as53046.anchors.atlas.ripe.net | 143.208.160.10 | probe: 7019


###Conclusão do Passo 2— Identificação de Anchors candidatas a destino

Foi consultado o endpoint `/anchors/` da API pública do RIPE Atlas.

Foram aplicados os filtros:

- `country=BR`: Anchors localizadas no Brasil;
- `is_disabled=false`: Anchors que não estão marcadas como desativadas.

Para cada Anchor são relevantes, inicialmente:

- `id`: identificador da Anchor;
- `fqdn`: nome da Anchor;
- `ip_v4`: endereço IPv4;
- `country`: país;
- `city`: cidade;
- `probe`: Probe ID hospedado na Anchor.

A Anchor será inicialmente tratada como um possível destino.

A existência de uma Anchor não implica, por si só, que uma Measurement
pública adequada já tenha sido selecionada. Essa associação será
investigada na próxima etapa.



```
                    DESTINO CANDIDATO
                           │
                     Anchor 2334
                    /           \
                  FQDN           IPv4
                    \
                     hospeda
                       │
                   Probe 6891
```



##PASSO 3 — Escolher uma Anchor como destino e localizar medições públicas de ping

* Como já escolhemos uma anchors como DESTINO temos identificar se existe algums Measurement publica de ping que tenha essa Anchor como alvo.

* Para realizar esta busca vamos usar `fqdn` e `ip_v4`. Eles são informações que podem aparecer como `target/target_id` em uma Measurement.

###Criar as variáveis da Anchor escolhida.

In [16]:
anchor_id = primeira_anchor.get("id")
anchor_fqdn = primeira_anchor.get("fqdn")
anchor_ip = primeira_anchor.get("ip_v4")

print("Anchor ID:", anchor_id)
print("FQDN:", anchor_fqdn)
print("IPv4:", anchor_ip)

Anchor ID: 2334
FQDN: br-bhz-as10417-client.anchors.atlas.ripe.net
IPv4: 150.164.1.222


**O que precisamos procurar?**

O endpoint `/api/v2/measurements/` fornece as seguintes informações: `id`, `type`, `target`, `target_ip`, `af`, `is_public`, `status`.

Portanto, desejamos que uma Measurements tenha estas características:
```
type      = ping
target    = nossa Anchor
af        = 4
is_public = true
```




###Criar a URL de Measurements

In [17]:
url_measurements = f"{BASE_URL}/measurements/"

print(url_measurements)

https://atlas.ripe.net/api/v2/measurements/


###Procurar pelo FQDN da Anchor

In [18]:
params = {
    "target": anchor_fqdn,
    "type": "ping",
    "af": 4,
    "is_public": "true"
}

response = requests.get(
    url_measurements,
    params=params,
    timeout=30
)

print("Status HTTP:", response.status_code)
print("URL consultada:")
print(response.url)

Status HTTP: 200
URL consultada:
https://atlas.ripe.net/api/v2/measurements/?target=br-bhz-as10417-client.anchors.atlas.ripe.net&type=ping&af=4&is_public=true


###Vamos examinar a estrutura da resposta

In [19]:
dados_measurements = response.json()
dados_measurements.keys()

dict_keys(['count', 'next', 'previous', 'results'])

###Quantidade de medições encontradas para o target

In [20]:
print("Measurements encontradas:", dados_measurements["count"])

Measurements encontradas: 270


> Se a quantidade de Measurement for 0, a Anchor é inválida. Portanto, este destino deve ser descartado.

###Viasualizar a primeira Measurement

In [21]:
primeira_measurement = dados_measurements["results"][0]
primeira_measurement

{'af': 4,
 'auto_topup': False,
 'auto_topup_prb_days_off': 7,
 'auto_topup_prb_similarity': 0.5,
 'creation_time': 1604946955,
 'credits_per_result': 3,
 'description': 'Anchoring Mesh Measurement: Ping IPv4 for anchor br-bhz-as10417-client.anchors.atlas.ripe.net',
 'estimated_results_per_day': 380880,
 'group': 'https://atlas.ripe.net/api/v2/measurements/groups/28095696/',
 'group_id': 28095696,
 'id': 28095697,
 'in_wifi_group': False,
 'include_probe_id': None,
 'interval': 240,
 'is_all_scheduled': True,
 'is_oneoff': False,
 'is_public': True,
 'packet_interval': 1000,
 'packets': 3,
 'participant_count': 1058,
 'probes_requested': 3973,
 'probes_scheduled': 1058,
 'resolve_on_probe': False,
 'resolved_ips': ['150.164.1.222'],
 'result': 'https://atlas.ripe.net/api/v2/measurements/28095697/results/',
 'size': 64,
 'spread': None,
 'start_time': 1604946955,
 'status': {'name': 'Ongoing', 'id': 2, 'when': None},
 'stop_time': None,
 'tags': ['anchoring', 'br-bhz-as10417-client', '6

###Extraindo o que é importante para o projeto

In [24]:
print("Measurement ID:", primeira_measurement.get("id"))
print("Tipo:", primeira_measurement.get("type"))
print("Target:", primeira_measurement.get("target"))
print("Target IP:", primeira_measurement.get("target_ip"))
print("AF:", primeira_measurement.get("af"))
print("Pública:", primeira_measurement.get("is_public"))
print("Status:", primeira_measurement.get("status"))

Measurement ID: 28095697
Tipo: ping
Target: br-bhz-as10417-client.anchors.atlas.ripe.net
Target IP: 150.164.1.222
AF: 4
Pública: True
Status: {'name': 'Ongoing', 'id': 2, 'when': None}


Agora temos:
```
Measurement Y
      │
      │ mede
      ▼
Anchor X
```




###Confimar se DESTINO correponde a Anchor

In [25]:
print("Anchor FQDN:")
print(anchor_fqdn)

print("\nTarget da Measurement:")
print(primeira_measurement.get("target"))

print("\nAnchor IPv4:")
print(anchor_ip)

print("\nTarget IP da Measurement:")
print(primeira_measurement.get("target_ip"))

Anchor FQDN:
br-bhz-as10417-client.anchors.atlas.ripe.net

Target da Measurement:
br-bhz-as10417-client.anchors.atlas.ripe.net

Anchor IPv4:
150.164.1.222

Target IP da Measurement:
150.164.1.222


###Quanfo a busca pelo FQDN retorna 0

* Não troque o Anchor imediatamente. Vamos fazer um teste como IPv4

In [26]:
params_ip = {
    "target": anchor_ip,
    "type": "ping",
    "af": 4,
    "is_public": "true"
}

response_ip = requests.get(
    url_measurements,
    params=params_ip,
    timeout=30
)

dados_measurements_ip = response_ip.json()

print("Status HTTP:", response_ip.status_code)
print("Measurements encontradas:", dados_measurements_ip["count"])
print(response_ip.url)

Status HTTP: 200
Measurements encontradas: 1141
https://atlas.ripe.net/api/v2/measurements/?target=150.164.1.222&type=ping&af=4&is_public=true


###Como escolher a Measurement

In [27]:
for m in dados_measurements["results"]:
    print(
        "ID:", m.get("id"),
        "| tipo:", m.get("type"),
        "| af:", m.get("af"),
        "| pública:", m.get("is_public"),
        "| status:", m.get("status"),
        "| target:", m.get("target"),
        "| target_ip:", m.get("target_ip")
    )

ID: 28095697 | tipo: ping | af: 4 | pública: True | status: {'name': 'Ongoing', 'id': 2, 'when': None} | target: br-bhz-as10417-client.anchors.atlas.ripe.net | target_ip: 150.164.1.222
ID: 28097617 | tipo: ping | af: 4 | pública: True | status: {'name': 'Ongoing', 'id': 2, 'when': None} | target: br-bhz-as10417-client.anchors.atlas.ripe.net | target_ip: 150.164.1.222
ID: 28101766 | tipo: ping | af: 4 | pública: True | status: {'name': 'Stopped', 'id': 4, 'when': 1604971502} | target: br-bhz-as10417-client.anchors.atlas.ripe.net | target_ip: 150.164.1.222
ID: 28304466 | tipo: ping | af: 4 | pública: True | status: {'name': 'Stopped', 'id': 4, 'when': 1606753802} | target: br-bhz-as10417-client.anchors.atlas.ripe.net | target_ip: 150.164.1.222
ID: 28305022 | tipo: ping | af: 4 | pública: True | status: {'name': 'Stopped', 'id': 4, 'when': 1606767902} | target: br-bhz-as10417-client.anchors.atlas.ripe.net | target_ip: 150.164.1.222
ID: 28329294 | tipo: ping | af: 4 | pública: True | statu

Temos que:
```
Measurement
│
├── type = ping
├── af = 4
├── is_public = True
├── target = Anchor escolhida
└── target_ip = IPv4 da Anchor
```
Vamos observar o `status`




In [28]:
primeira_measurement.get("status")

{'name': 'Ongoing', 'id': 2, 'when': None}

Se:
```
Ongoing
   ↓
Measurement continua sendo executada
```


Se:
```
Stopped
   ↓
Measurement já terminou
```


Vamos fazer uma análise antes da escolha: Vamos guardar apenas algumas características de uma measurement com status=on going

In [30]:
measurement_id = primeira_measurement.get("id")
measurement_target = primeira_measurement.get("target")
measurement_target_ip = primeira_measurement.get("target_ip")

print("Measurement selecionada para análise:", measurement_id)
print("Destino:", measurement_target)
print("Destino IPv4:", measurement_target_ip)

Measurement selecionada para análise: 28095697
Destino: br-bhz-as10417-client.anchors.atlas.ripe.net
Destino IPv4: 150.164.1.222


Temos:
```
                      RIPE ATLAS

                    Anchor Brasil
                    ID = anchor_id
                        ▲
                        │
                        │ target
                        │
                Measurement pública
                 ID = measurement_id
                 type = ping
                 af = 4
                        ▲
                        │
                        │ ???
                        │
                   PROBES
```

Portanto,
 * DESTINO OK
 * MEASUREMENT OK
 * Mas falta definir a ORIGEM.

###Escolhendo as probes

* Quais pobes realmente produziram resultados nessa Measurement?
* Cada resultado (registro) de Measurement possui um `prb_id` que identifica quem realizou aquela medição.

In [31]:
print("Quantidade:", dados_measurements["count"])

for m in dados_measurements["results"][:10]:
    print(
        m.get("id"),
        "|", m.get("type"),
        "| AF:", m.get("af"),
        "|", m.get("status"),
        "|", m.get("target"),
        "|", m.get("target_ip")
    )

Quantidade: 270
28095697 | ping | AF: 4 | {'name': 'Ongoing', 'id': 2, 'when': None} | br-bhz-as10417-client.anchors.atlas.ripe.net | 150.164.1.222
28097617 | ping | AF: 4 | {'name': 'Ongoing', 'id': 2, 'when': None} | br-bhz-as10417-client.anchors.atlas.ripe.net | 150.164.1.222
28101766 | ping | AF: 4 | {'name': 'Stopped', 'id': 4, 'when': 1604971502} | br-bhz-as10417-client.anchors.atlas.ripe.net | 150.164.1.222
28304466 | ping | AF: 4 | {'name': 'Stopped', 'id': 4, 'when': 1606753802} | br-bhz-as10417-client.anchors.atlas.ripe.net | 150.164.1.222
28305022 | ping | AF: 4 | {'name': 'Stopped', 'id': 4, 'when': 1606767902} | br-bhz-as10417-client.anchors.atlas.ripe.net | 150.164.1.222
28329294 | ping | AF: 4 | {'name': 'Stopped', 'id': 4, 'when': 1607063102} | br-bhz-as10417-client.anchors.atlas.ripe.net | 150.164.1.222
28329854 | ping | AF: 4 | {'name': 'Stopped', 'id': 4, 'when': 1607088303} | br-bhz-as10417-client.anchors.atlas.ripe.net | 150.164.1.222
28336470 | ping | AF: 4 | {'na


```
Measurement ID: 28095697
        │
        ├── tipo = ping
        ├── IPv4
        ├── pública
        ├── status = Ongoing
        │
        └── destino
               │
               ▼
     br-bhz-as10417-client...
           150.164.1.222
    ```



* Encontramos 2 medições com status `Ongoing`:
```
28095697 → Ongoing
28097617 → Ongoing
```
* O próximo passo é escolher qual destas Measurement escolher.



###Conclusão do Passo 3 -Associação entre Anchor e Measurement

Foi escolhida uma Anchor candidata a destino.

A partir de seu FQDN e endereço IPv4, foram pesquisadas
Measurements públicas existentes no RIPE Atlas.

Foram considerados inicialmente os seguintes critérios:

- tipo: `ping`;
- família de endereço: IPv4 (`af=4`);
- medição pública (`is_public=true`);
- destino correspondente ao FQDN ou IPv4 da Anchor.

Neste ponto, a Measurement representa a campanha de medição
que possui a Anchor escolhida como destino.

Ainda não foram escolhidas as probes de origem.

As origens deverão ser identificadas posteriormente a partir
dos `prb_id` efetivamente presentes nos resultados da Measurement.

##PASSO 4 — Descobrir quem realmente mede essa Anchor

Temos definido: `DETINO` que é Anchor `BR` e a Measurement pública.
O que falta definir: `ORIGEM`

###Vamos escolher um measurement_id para inspecionar

In [32]:
measurement_id = 28095697

###Vamos consultar uma janela pequena de 30 minutos

In [33]:
from datetime import datetime, timezone, timedelta

inicio = datetime(
    2026, 9, 21, 12, 0, 0,
    tzinfo=timezone.utc
)

fim = inicio + timedelta(minutes=30)

print(inicio)
print(fim)

2026-09-21 12:00:00+00:00
2026-09-21 12:30:00+00:00


###Converter para Unix timestamp

* A API usa `start` e `stop`.

In [34]:
start = int(inicio.timestamp())
stop = int(fim.timestamp())

print("start:", start)
print("stop:", stop)

start: 1789992000
stop: 1789993800


###Montando o endpoint de resultados

In [35]:
url_results = (
    f"{BASE_URL}/measurements/"
    f"{measurement_id}/results/"
)

print(url_results)

https://atlas.ripe.net/api/v2/measurements/28095697/results/


###Fazer uma primeira consulta sem escolher probes

In [37]:
params = {
    "start": start,
    "stop": stop,
    "format": "json"
}

response = requests.get(
    url_results,
    params=params,
    timeout=30
)

print("Status:", response.status_code)
print(response.url)

Status: 200
https://atlas.ripe.net/api/v2/measurements/28095697/results/?start=1789992000&stop=1789993800&format=json


###Converter o resultado em JSON

In [38]:
resultados = response.json()

In [39]:
type(resultados)

list

In [40]:
len(resultados)

7336

* quantidade de observações no intervalo de tempo

###Ver o primeiro resultado

In [41]:
resultados[0]

{'fw': 5130,
 'mver': '2.6.4',
 'lts': 54,
 'dst_name': '150.164.1.222',
 'af': 4,
 'dst_addr': '150.164.1.222',
 'src_addr': '103.141.79.162',
 'proto': 'ICMP',
 'ttl': 42,
 'size': 64,
 'result': [{'rtt': 230.458996}, {'rtt': 230.531635}, {'rtt': 231.767101}],
 'dup': 0,
 'rcvd': 3,
 'sent': 3,
 'min': 230.458996,
 'max': 231.767101,
 'avg': 230.91924400000002,
 'msm_id': 28095697,
 'prb_id': 7755,
 'timestamp': 1789992243,
 'msm_name': 'Ping',
 'from': '103.141.79.162',
 'type': 'ping',
 'group_id': 28095696,
 'step': 240,
 'stored_timestamp': 1789992496}

###Extrair apenas os campos importantes

In [42]:
primeiro_resultado = resultados[0]

print("Measurement:", primeiro_resultado.get("msm_id"))
print("Probe:", primeiro_resultado.get("prb_id"))
print("Destino:", primeiro_resultado.get("dst_addr"))
print("Timestamp:", primeiro_resultado.get("timestamp"))
print("RTT médio:", primeiro_resultado.get("avg"))
print("Enviados:", primeiro_resultado.get("sent"))
print("Recebidos:", primeiro_resultado.get("rcvd"))

Measurement: 28095697
Probe: 7755
Destino: 150.164.1.222
Timestamp: 1789992243
RTT médio: 230.91924400000002
Enviados: 3
Recebidos: 3


Temos:
```
Probe 12345
     │
     │ ping
     ▼
150.164.1.222
Anchor Belo Horizonte
```

Esse é o primeiro par origem-destino efetivamente observado.


###Descobrir todos os prb_id diferentes

In [43]:
probe_ids = sorted({
    resultado["prb_id"]
    for resultado in resultados
    if "prb_id" in resultado
})

print(probe_ids)

[6251, 6313, 6316, 6317, 6320, 6323, 6324, 6325, 6326, 6327, 6328, 6329, 6330, 6331, 6332, 6333, 6334, 6335, 6336, 6342, 6346, 6348, 6349, 6350, 6351, 6352, 6358, 6360, 6363, 6364, 6366, 6370, 6372, 6373, 6374, 6379, 6380, 6382, 6385, 6387, 6389, 6390, 6392, 6393, 6396, 6398, 6402, 6407, 6408, 6410, 6411, 6413, 6414, 6417, 6420, 6424, 6425, 6426, 6427, 6428, 6429, 6431, 6432, 6434, 6435, 6436, 6437, 6438, 6439, 6441, 6443, 6446, 6448, 6450, 6451, 6452, 6453, 6454, 6456, 6475, 6476, 6477, 6480, 6482, 6485, 6488, 6490, 6493, 6494, 6496, 6498, 6503, 6505, 6506, 6507, 6509, 6511, 6512, 6513, 6514, 6515, 6516, 6520, 6521, 6522, 6524, 6525, 6526, 6533, 6534, 6535, 6538, 6540, 6541, 6543, 6544, 6545, 6547, 6548, 6550, 6552, 6553, 6554, 6558, 6559, 6561, 6562, 6563, 6564, 6565, 6567, 6568, 6569, 6571, 6574, 6576, 6577, 6580, 6581, 6583, 6585, 6588, 6589, 6590, 6594, 6597, 6598, 6599, 6601, 6602, 6604, 6609, 6616, 6618, 6619, 6624, 6626, 6629, 6630, 6631, 6632, 6634, 6635, 6636, 6641, 6643, 664

In [45]:
print("Quantidade de probes:", len(probe_ids))

Quantidade de probes: 1000


Agora saímos disto:
```
Measurement 28095697
        │
        ▼
Anchor BR
```


Para:
```
Probe A ─────┐
Probe B ─────┤
Probe C ─────┼──► Measurement 28095697 ───► Anchor BR
Probe D ─────┘
```

No entanto, ainda não sabemos de quais países são essas probes.


###Consultar uma Probe individual

In [46]:
probe_id = probe_ids[0]

In [47]:
url_probe = f"{BASE_URL}/probes/{probe_id}/"

response_probe = requests.get(
    url_probe,
    timeout=30
)

print("Status:", response_probe.status_code)

Status: 200


In [48]:
probe = response_probe.json()
probe

{'address_v4': '193.65.45.86',
 'address_v6': '2001:998:13:1b2::2',
 'asn_v4': 719,
 'asn_v6': 719,
 'country_code': 'FI',
 'description': 'Elisa Oyj',
 'firmware_version': 5130,
 'first_connected': 1485247275,
 'geometry': {'type': 'Point', 'coordinates': [23.7595, 61.4875]},
 'id': 6251,
 'is_anchor': True,
 'is_public': True,
 'last_connected': 1791470952,
 'prefix_v4': '193.64.0.0/15',
 'prefix_v6': '2001:998::/29',
 'status': {'id': 1, 'name': 'Connected', 'since': '2026-10-06T12:52:39Z'},
 'status_since': 1791291159,
 'tags': [{'name': 'system: Anchor', 'slug': 'system-anchor'},
  {'name': 'system: Resolves A Correctly',
   'slug': 'system-resolves-a-correctly'},
  {'name': 'system: Resolves AAAA Correctly',
   'slug': 'system-resolves-aaaa-correctly'},
  {'name': 'system: IPv4 Works', 'slug': 'system-ipv4-works'},
  {'name': 'system: IPv6 Works', 'slug': 'system-ipv6-works'},
  {'name': 'system: IPv4 Capable', 'slug': 'system-ipv4-capable'},
  {'name': 'system: IPv6 Capable', 's

###Extrair os dados essenciais

In [49]:
print("Probe ID:", probe.get("id"))
print("País:", probe.get("country_code"))
print("É Anchor:", probe.get("is_anchor"))
print("Pública:", probe.get("is_public"))

Probe ID: 6251
País: FI
É Anchor: True
Pública: True


Temos:

```
Origem
Probe 6251
País: FI (Finlândia)
É Anchor: True
        │
        │ ping
        ▼
Destino
150.164.1.222
Anchor em Belo Horizonte            destino
   
```

Identificamos o primeiro par real: FI → BR (Probe 6251 → 150.164.1.222)

* Queremos construir pares de forma controlada. Ex:
```
BR → BR
BR → UY
BR → JP
DE → PT
```




#PASSO 4B — Filtrar somente Anchors como origem:

* Anchor será a Origem
* logo: anchors_only=true


In [50]:
params = {
    "start": start,
    "stop": stop,
    "anchors_only": "true",
    "public_only": "true",
    "format": "json"
}

response = requests.get(
    url_results,
    params=params,
    timeout=30
)

print("Status:", response.status_code)

Status: 200


In [51]:
resultados_anchors = response.json()

print("Quantidade de resultados:", len(resultados_anchors))

Quantidade de resultados: 7336


###Extrair os IDs únicos

In [52]:
anchor_probe_ids = sorted({
    resultado["prb_id"]
    for resultado in resultados_anchors
    if "prb_id" in resultado
})

print("Quantidade de Anchors de origem:", len(anchor_probe_ids))
print(anchor_probe_ids[:20])

Quantidade de Anchors de origem: 1000
[6251, 6313, 6316, 6317, 6320, 6323, 6324, 6325, 6326, 6327, 6328, 6329, 6330, 6331, 6332, 6333, 6334, 6335, 6336, 6342]


* Não vamos escolher 4 IDs simplesmente. Quermos fazer escolhas planejadas.
* Exemplo:
```
quero até 4 Anchors do Brasil
quero até 4 Anchors da Alemanha
quero até 4 Anchors da Argentina
```



#PASSO 4C — Descobrir o país de cada Anchor de origem

* Consulte os metadados das Anchors

In [53]:
amostra_ids = anchor_probe_ids[:20]

In [54]:
origens = []

for probe_id in amostra_ids:

    url_probe = f"{BASE_URL}/probes/{probe_id}/"

    response_probe = requests.get(
        url_probe,
        timeout=30
    )

    if response_probe.status_code == 200:

        probe = response_probe.json()

        origens.append({
            "probe_id": probe.get("id"),
            "country_code": probe.get("country_code"),
            "is_anchor": probe.get("is_anchor"),
            "is_public": probe.get("is_public")
        })

In [55]:
origens

[{'probe_id': 6251,
  'country_code': 'FI',
  'is_anchor': True,
  'is_public': True},
 {'probe_id': 6313,
  'country_code': 'DE',
  'is_anchor': True,
  'is_public': True},
 {'probe_id': 6316,
  'country_code': 'IT',
  'is_anchor': True,
  'is_public': True},
 {'probe_id': 6317,
  'country_code': 'CH',
  'is_anchor': True,
  'is_public': True},
 {'probe_id': 6320,
  'country_code': 'FR',
  'is_anchor': True,
  'is_public': True},
 {'probe_id': 6323,
  'country_code': 'PL',
  'is_anchor': True,
  'is_public': True},
 {'probe_id': 6324,
  'country_code': 'LT',
  'is_anchor': True,
  'is_public': True},
 {'probe_id': 6325,
  'country_code': 'AT',
  'is_anchor': True,
  'is_public': True},
 {'probe_id': 6326,
  'country_code': 'UA',
  'is_anchor': True,
  'is_public': True},
 {'probe_id': 6327,
  'country_code': 'HR',
  'is_anchor': True,
  'is_public': True},
 {'probe_id': 6328,
  'country_code': 'PT',
  'is_anchor': True,
  'is_public': True},
 {'probe_id': 6329,
  'country_code': 'IT',

###Mostrar de maneira mais legível

In [56]:
for origem in origens:
    print(
        origem["probe_id"],
        "|",
        origem["country_code"],
        "| Anchor:",
        origem["is_anchor"],
        "| Pública:",
        origem["is_public"]
    )

6251 | FI | Anchor: True | Pública: True
6313 | DE | Anchor: True | Pública: True
6316 | IT | Anchor: True | Pública: True
6317 | CH | Anchor: True | Pública: True
6320 | FR | Anchor: True | Pública: True
6323 | PL | Anchor: True | Pública: True
6324 | LT | Anchor: True | Pública: True
6325 | AT | Anchor: True | Pública: True
6326 | UA | Anchor: True | Pública: True
6327 | HR | Anchor: True | Pública: True
6328 | PT | Anchor: True | Pública: True
6329 | IT | Anchor: True | Pública: True
6330 | FI | Anchor: True | Pública: True
6331 | DE | Anchor: True | Pública: True
6332 | DE | Anchor: True | Pública: True
6333 | FR | Anchor: True | Pública: True
6334 | AT | Anchor: True | Pública: True
6335 | RU | Anchor: True | Pública: True
6336 | DE | Anchor: True | Pública: True
6342 | NL | Anchor: True | Pública: True


Cada seta é um possível par:



```
Anchor FI ─────┐
Anchor DE ─────┤
Anchor NL ─────┼────► Anchor BR
Anchor JP ─────┤
Anchor US ─────┘

três fluxo: (fluxo_id = probe_id | dst_addr)

6251 → 150.164.1.222
7001 → 150.164.1.222
8001 → 150.164.1.222
```

1. Temos que se o DESTINO atual é o Brasil: (150.164.1.222)
2. E encontramos as probes:


```
Probe 6251 → FI
Probe 7001 → DE
Probe 8001 → BR
Probe 8002 → BR
Probe 8003 → BR
```
3. Selecionamos as probes:


```
BR → BR

8001 → 150.164.1.222
8002 → 150.164.1.222
8003 → 150.164.1.222
```

E o próximo passo é escolher uma anchor no Japão por exemplo e repetir o processo novamente.





###Conclusão do Passo 4 — Identificação das origens

A consulta dos resultados da Measurement mostrou que muitas probes
participam da medição do destino escolhido.

Cada resultado possui um `prb_id`, que identifica a origem efetiva
da medição.

Como o projeto utilizará Anchors como origem, a consulta pode ser
restringida com:

- `anchors_only=true`;
- `public_only=true`.

Após identificar os `prb_id`, o endpoint `/probes/{id}/` permite
obter o país (`country_code`) e confirmar se a origem é uma Anchor.

Assim, os pares origem-destino são definidos somente entre origens
que efetivamente produziram resultados para a Measurement.

# COLETA DE DADOS UTILIZANDO OS PARES

imports + PASTA_RAW

In [57]:
import sys
import json
from pathlib import Path
from datetime import datetime, timezone, timedelta

import pandas as pd
import requests

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    PASTA_PROJETO = Path("/content/drive/MyDrive/ripe_atlas")
else:
    PASTA_PROJETO = Path("./ripe_atlas")

PASTA_RAW = PASTA_PROJETO / "raw"
PASTA_RAW.mkdir(parents=True, exist_ok=True)

print("Arquivos raw serão salvos em:", PASTA_RAW)

Mounted at /content/drive
Arquivos raw serão salvos em: /content/drive/MyDrive/ripe_atlas/raw


0. Pré-requisito

In [58]:
def obter_metadados_medicao(measurement_id, timeout_requisicao):
    url_metadados = f"https://atlas.ripe.net/api/v2/measurements/{measurement_id}/"
    response = requests.get(url_metadados, timeout=timeout_requisicao)
    response.raise_for_status()
    info = response.json()
    return {
        "is_oneoff": info.get("is_oneoff"),
        "interval": info.get("interval"),
        "packets": info.get("packets"),
        "start_time": info.get("start_time"),
        "stop_time": info.get("stop_time"),
    }

1. Parâmetros e verificação de que a medição é contínua

In [59]:
JANELA_HORAS = 168
TIMEOUT_REQUISICAO = 60
FORMATO_TABELAR = "csv"

metadados_medicao = obter_metadados_medicao(measurement_id, TIMEOUT_REQUISICAO)
print("is_oneoff:", metadados_medicao["is_oneoff"])
print("interval:", metadados_medicao["interval"])

assert metadados_medicao["is_oneoff"] is False, (
    "Essa medição é one-off — não serve para coletar pares ao longo do tempo."
)

is_oneoff: False
interval: 240


2. Coletar os resultados dos pares (uma chamada só)

In [60]:
def coletar_resultados_pares(measurement_id, prb_ids, janela_horas, timeout_requisicao, pasta_raw, formato_tabelar="csv"):
    fim_utc = datetime.now(timezone.utc)
    inicio_utc = fim_utc - timedelta(hours=janela_horas)

    url = f"https://atlas.ripe.net/api/v2/measurements/{measurement_id}/results/"
    parametros = {
        "start": int(inicio_utc.timestamp()),
        "stop": int(fim_utc.timestamp()),
        "probe_ids": ",".join(str(p) for p in prb_ids),
        "format": "json",
    }

    print("URL:", url)
    print("Início (UTC):", inicio_utc.isoformat())
    print("Fim (UTC):", fim_utc.isoformat())
    print("Pares (prb_id) solicitados:", len(prb_ids))

    instante_requisicao_utc = datetime.now(timezone.utc)
    response = requests.get(url, params=parametros, timeout=timeout_requisicao)
    response.raise_for_status()
    dados_json = response.json()

    print("Status HTTP:", response.status_code)
    print("Quantidade de registros recebidos:", len(dados_json))

    if len(dados_json) == 0:
        raise RuntimeError("Nenhum registro retornado — revise a janela ou os prb_ids.")

    df_raw = pd.json_normalize(dados_json, sep="_")
    display(df_raw.head())

    timestamp_arquivo = instante_requisicao_utc.strftime("%Y%m%dT%H%M%SZ")
    nome_base = f"ripe_atlas_pares_m{measurement_id}_{timestamp_arquivo}"

    caminho_json = pasta_raw / f"{nome_base}.json"
    with caminho_json.open("w", encoding="utf-8") as arquivo:
        json.dump(dados_json, arquivo, ensure_ascii=False, indent=2)

    caminho_tabela = pasta_raw / f"{nome_base}.{formato_tabelar}"
    if formato_tabelar == "csv":
        df_raw.to_csv(caminho_tabela, index=False, encoding="utf-8")
    else:
        df_raw.to_parquet(caminho_tabela, index=False)

    metadados = {
        "measurement_id": measurement_id,
        "is_oneoff": metadados_medicao["is_oneoff"],
        "interval": metadados_medicao["interval"],
        "quantidade_pares_solicitados": len(prb_ids),
        "pares_prb_ids": list(prb_ids),
        "url": url,
        "parametros": parametros,
        "inicio_utc": inicio_utc.isoformat(),
        "fim_utc": fim_utc.isoformat(),
        "coletado_em_utc": instante_requisicao_utc.isoformat(),
        "status_http": response.status_code,
        "quantidade_registros": len(dados_json),
        "arquivo_json": caminho_json.name,
        "arquivo_tabelar": caminho_tabela.name,
    }
    caminho_metadados = pasta_raw / f"{nome_base}_metadata.json"
    with caminho_metadados.open("w", encoding="utf-8") as arquivo:
        json.dump(metadados, arquivo, ensure_ascii=False, indent=2)

    return {"df_raw": df_raw, "dados_json": dados_json, "caminho_json": caminho_json,
            "caminho_tabela": caminho_tabela, "caminho_metadados": caminho_metadados}


resultado_pares = coletar_resultados_pares(
    measurement_id, anchor_probe_ids[:20], JANELA_HORAS, TIMEOUT_REQUISICAO, PASTA_RAW
)

URL: https://atlas.ripe.net/api/v2/measurements/28095697/results/
Início (UTC): 2026-10-01T14:53:43.390784+00:00
Fim (UTC): 2026-10-08T14:53:43.390784+00:00
Pares (prb_id) solicitados: 20
Status HTTP: 200
Quantidade de registros recebidos: 50250


,fw,mver,lts,dst_name,af,dst_addr,src_addr,proto,ttl,size,...,avg,msm_id,prb_id,timestamp,msm_name,from,type,group_id,step,stored_timestamp
0,5130,2.6.4,34,150.164.1.222,4,150.164.1.222,213.212.129.68,ICMP,44.0,64,...,220.962967,28095697,6316,1790866556,Ping,213.212.129.68,ping,28095696,240,1790866572
1,5130,2.6.4,34,150.164.1.222,4,150.164.1.222,130.59.80.2,ICMP,47.0,64,...,135.169023,28095697,6317,1790866559,Ping,130.59.80.2,ping,28095696,240,1790866585
2,5130,2.6.4,19,150.164.1.222,4,150.164.1.222,94.136.2.190,ICMP,45.0,64,...,232.624755,28095697,6325,1790866551,Ping,94.136.2.190,ping,28095696,240,1790866588
3,5130,2.6.4,27,150.164.1.222,4,150.164.1.222,91.202.128.78,ICMP,41.0,64,...,174.253547,28095697,6326,1790866553,Ping,91.202.128.78,ping,28095696,240,1790866596
4,5130,2.6.4,15,150.164.1.222,4,150.164.1.222,161.53.251.28,ICMP,46.0,64,...,140.074588,28095697,6327,1790866553,Ping,161.53.251.28,ping,28095696,240,1790866589


3. Validação

In [61]:
def validar_pares(df_raw, dados_json):
    por_sonda = df_raw.groupby("prb_id").size().sort_values(ascending=False)
    print("Registros por sonda (par):")
    print(por_sonda)

    duplicatas = df_raw.duplicated(subset=["prb_id", "timestamp"]).sum()
    sondas_timeout_total = (df_raw["rcvd"] == 0).sum()
    pacotes_perdidos = sum(1 for r in dados_json for p in r.get("result", []) if "x" in p)

    print("\nPares distintos:", df_raw["prb_id"].nunique())
    print("Duplicatas:", duplicatas)
    print("Registros com timeout total:", sondas_timeout_total)
    print("Pacotes individuais perdidos:", pacotes_perdidos)

    return por_sonda


contagem_por_par = validar_pares(resultado_pares["df_raw"], resultado_pares["dados_json"])

Registros por sonda (par):
prb_id
6330    2520
6332    2520
6251    2519
6313    2519
6323    2519
6329    2519
6333    2518
6331    2518
6324    2518
6316    2517
6317    2517
6326    2517
6342    2517
6327    2517
6328    2517
6320    2516
6325    2515
6336    2510
6335    2487
6334    2450
dtype: int64

Pares distintos: 20
Duplicatas: 0
Registros com timeout total: 44
Pacotes individuais perdidos: 269


4. Transformar para o Contrato de Dados e salvar o CSV final

In [62]:
def transformar_para_contrato(dados_json, destino_fqdn):
    registros = []
    for sonda in dados_json:
        enviados, recebidos = sonda["sent"], sonda["rcvd"]
        perda_pct = ((enviados - recebidos) / enviados) * 100 if enviados > 0 else 100
        rtts = [r["rtt"] for r in sonda.get("result", []) if "rtt" in r]
        if len(rtts) > 1:
            jitter = sum(abs(rtts[i] - rtts[i-1]) for i in range(1, len(rtts))) / (len(rtts) - 1)
        else:
            jitter = 0.0

        registros.append({
            "timestamp": pd.to_datetime(sonda["timestamp"], unit="s", utc=True),
            "par_id": f"{sonda['prb_id']}->{destino_fqdn}",
            "prb_id": sonda["prb_id"],
            "ip": sonda["src_addr"],
            "latencia_ms": round(sonda.get("avg", 0), 2),
            "perda_pacotes_pct": round(perda_pct, 2),
            "jitter_ms": round(jitter, 2),
        })
    return pd.DataFrame(registros)


def classificar_status_relativo(df):
    estatisticas = (
        df[df["perda_pacotes_pct"] == 0]
        .groupby("par_id")["latencia_ms"]
        .agg(latencia_media="mean", latencia_desvio="std")
    )
    df = df.merge(estatisticas, on="par_id", how="left")

    def status(linha):
        if linha["perda_pacotes_pct"] > 0:
            return "FALHA"
        limite_risco = linha["latencia_media"] + linha["latencia_desvio"]
        limite_falha = linha["latencia_media"] + 3 * linha["latencia_desvio"]
        if linha["latencia_ms"] >= limite_falha:
            return "FALHA"
        elif linha["latencia_ms"] >= limite_risco:
            return "RISCO"
        else:
            return "OK"

    df["status_real"] = df.apply(status, axis=1)
    return df.drop(columns=["latencia_media", "latencia_desvio"])


df_contrato = transformar_para_contrato(resultado_pares["dados_json"], anchor_fqdn)
df_contrato = classificar_status_relativo(df_contrato)

caminho_csv_final = PASTA_RAW / "dataset_pares_contrato.csv"
df_contrato.to_csv(caminho_csv_final, index=False, encoding="utf-8")
print("CSV final salvo em:", caminho_csv_final)
df_contrato["status_real"].value_counts()

CSV final salvo em: /content/drive/MyDrive/ripe_atlas/raw/dataset_pares_contrato.csv


,count
status_real,
OK,46980
RISCO,2547
FALHA,723
